# 01 · FLUX.2 [klein] 本地出图与参数消融

**硬件**：🟡 消费级 GPU（4B 模型 bf16 约 13GB VRAM；`enable_model_cpu_offload` 后更低。Apple Silicon 的 mps 也能跑，速度较慢）

## 本 notebook 你将学到

1. 跑通 Apache 2.0 的 FLUX.2 [klein]——2026 年初发布的开源快速出图代表
2. 三个核心参数的**消融实验**：steps、guidance_scale、seed，建立"参数→画面"直觉
3. base 版（CFG 完整训练）与蒸馏版的关系（对应 [theory.md](../theory.md) 第 3 节的步数蒸馏）
4. Flow matching 采样的中间状态可视化：亲眼看"噪声一步步变成图"

> 版本提示（2026-08）：以 [FLUX.2-klein-4B 模型卡](https://huggingface.co/black-forest-labs/FLUX.2-klein-4B) 为准；首次运行需接受 license 并 `huggingface-cli login`。

In [ ]:
%pip install -q "diffusers>=0.36" transformers accelerate safetensors torch matplotlib

In [ ]:
import torch
from diffusers import Flux2KleinPipeline

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

# base 版：未做 guidance 蒸馏，参数效果最"教科书"，适合做消融实验
# 追求速度可换 "black-forest-labs/FLUX.2-klein-4B"（蒸馏版，少步数出图）
pipe = Flux2KleinPipeline.from_pretrained(
    "black-forest-labs/FLUX.2-klein-base-4B", torch_dtype=torch.bfloat16
)
if device == "cuda":
    pipe.enable_model_cpu_offload()   # 显存不够时的救星
else:
    pipe = pipe.to(device)

## 1. 第一张图

In [ ]:
prompt = (
    "A cozy bookstore cafe at dusk, warm window light, a cat sleeping on a stack of books, "
    "a chalkboard sign that says 'Multimodal 101', photorealistic, 35mm"
)

image = pipe(
    prompt=prompt,
    height=768, width=768,
    guidance_scale=4.0,
    num_inference_steps=28,
    generator=torch.Generator(device="cpu").manual_seed(42),
).images[0]
image.save("first.png")
image

检查两件事：**文字渲染**（招牌上的 'Multimodal 101' 拼对了吗？）和**光影一致性**（窗光方向统一吗？）——这两项是新一代模型甩开 SD1.5 时代最直观的地方。

## 2. 消融一：采样步数

Flow matching 学的是"噪声→图像"的速度场，步数 = 沿轨迹积分的精度。看质量什么时候饱和——多出来的步数纯属烧钱。

In [ ]:
import matplotlib.pyplot as plt
import time

steps_list = [4, 8, 16, 28, 50]
fig, axes = plt.subplots(1, len(steps_list), figsize=(20, 4.5))
for ax, steps in zip(axes, steps_list):
    t0 = time.perf_counter()
    img = pipe(prompt=prompt, height=512, width=512, guidance_scale=4.0,
               num_inference_steps=steps,
               generator=torch.Generator(device="cpu").manual_seed(42)).images[0]
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"{steps} steps / {time.perf_counter()-t0:.1f}s")
plt.tight_layout(); plt.show()

# base 版在 4-8 步时应该明显劣化——而蒸馏版 klein 在 4 步就能出好图，
# 这个差距就是"步数蒸馏"技术的价值（theory.md 第 3 节）。

## 3. 消融二：guidance scale

CFG 强度 = "多听 prompt 的话"。太低跑题，太高过饱和、构图僵硬。

In [ ]:
cfg_list = [1.0, 2.5, 4.0, 7.0, 12.0]
fig, axes = plt.subplots(1, len(cfg_list), figsize=(20, 4.5))
for ax, cfg in zip(axes, cfg_list):
    img = pipe(prompt=prompt, height=512, width=512, guidance_scale=cfg,
               num_inference_steps=28,
               generator=torch.Generator(device="cpu").manual_seed(42)).images[0]
    ax.imshow(img); ax.axis("off"); ax.set_title(f"cfg={cfg}")
plt.tight_layout(); plt.show()

## 4. 消融三：seed 与"抽卡"

同 prompt 不同 seed = 同一分布的不同采样。生产中"固定 seed + 微调 prompt"是可控迭代的基本功。

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, seed in zip(axes, [0, 1, 2, 3]):
    img = pipe(prompt=prompt, height=512, width=512, guidance_scale=4.0,
               num_inference_steps=28,
               generator=torch.Generator(device="cpu").manual_seed(seed)).images[0]
    ax.imshow(img); ax.axis("off"); ax.set_title(f"seed={seed}")
plt.tight_layout(); plt.show()

## 练习

1. **中文文字渲染压力测试**：让招牌写"多模态101"，对比 [Qwen-Image](../landscape.md)——中文渲染是它的主场。
2. 组合性测试（GenEval 风格）："a red cube on top of a blue sphere, to the left of a yellow cone"，数数摆对了几个。
3. 把同一组 prompt 发给 GPT Image 2 / Nano Banana API，做一次开源 vs 闭源盲评（拉上朋友投票）。
4. 试 img2img：用 `pipe(image=..., strength=0.6, ...)` 给你的照片换风格，为 04 章编辑做热身。